In [11]:
import numpy as np
import math
import os

In [15]:
# Signal to noise ratio, in discrete and continuous forms
disc_SNR_dB = [-5, 0, 5, 10, 15, 20]
cont_SNR_dB = np.random.uniform(-5, 20)

In [5]:
# RF signals
def generate_bits(bit_num):
    bits = np.random.randint(0, 2, bit_num)
    return bits

In [6]:
# BPSK
def bpsk_mod(bits):
    """Converts raw signals into BPSK modulated symbols
    0 -> -1
    1 -> +1    """
    bpsk_symbols = 2 * bits - 1
    return bpsk_symbols

In [13]:
def qpsk_mod(bits):
    """Converts raw signals into QPSK modulated symbols
    Considers a set of 2 bits
    sqrt(2) is used to normalize such that the average symbol power = 1
    00 -> 1 + j / sqrt(2)
    01 -> -1 + j/ sqrt(2)
    11 -> -1 -j/ sqrt(2)
    10 ->  1 - j/ sqrt(2)    """

    if len(bits)%2==0:
        pairs = bits.reshape(-1,2)

        qpsk_symbols = np.empty(len(pairs),dtype = np.complex64)

        for i, (bit1,bit2) in enumerate(pairs):
            # bit 1 -> complex part
            if bit1 == 0:
                imag = 1
            else:
                imag = -1

            # bit 2 -> real part    
            if bit2 == 0:
                real = 1
            else:
                real = -1
            qpsk_symbols[i] = (real + (imag*1j))/np.sqrt(2)
        return qpsk_symbols
    else :
        return None

In [8]:
def snr_transform(snr_db):
    """Converts SNR from dB to a linear form"""
    snr_linear = 10 ** (snr_db / 10)
    return snr_linear

In [9]:
def awgn(bpsk_symbols,snr_db):
    """Computes and Adds the required AWGN noise to the required bits"""

    snr_linear = snr_transform(snr_db)

    sigma = np.sqrt(1 / (2 * snr_linear)) # computes the standard deviation of the guassian dist.

    awgn_noise = np.random.normal(0,sigma,len(bpsk_symbols))
    
    received_signal = bpsk_symbols + awgn_noise
    
    return received_signal,awgn_noise

In [16]:
# Sample data
bit_num = 64
bits = generate_bits(bit_num)
symbols = qpsk_mod(bits)

snr_db = np.random.choice(disc_SNR_dB)
print("SNR (dB): ",snr_db)

snr_linear = snr_transform(snr_db)
received_signal,noise = awgn(symbols,snr_db)
received_signal

SNR (dB):  10


array([ 0.93645812-0.70710677j, -0.79182738+0.70710677j,
        0.61389216-0.70710677j,  0.85363802-0.70710677j,
       -0.8156958 +0.70710677j,  0.55835949+0.70710677j,
        1.12641633+0.70710677j,  0.57107993-0.70710677j,
        0.58168687+0.70710677j,  0.2992963 -0.70710677j,
        0.33872611-0.70710677j,  0.63412229+0.70710677j,
       -0.51820557+0.70710677j, -0.69687617-0.70710677j,
        0.65005911+0.70710677j, -0.82421825+0.70710677j,
        0.59893456-0.70710677j,  0.53645956-0.70710677j,
        1.06646956+0.70710677j, -0.59637379+0.70710677j,
       -0.95261544+0.70710677j, -1.02323725-0.70710677j,
        0.92071995+0.70710677j, -1.03557588-0.70710677j,
       -0.63491508+0.70710677j, -0.52127276+0.70710677j,
        0.34936279+0.70710677j, -0.52816557-0.70710677j,
        0.93195595-0.70710677j, -1.05568001-0.70710677j,
       -0.94387708+0.70710677j,  0.80170189-0.70710677j])

In [33]:
"""Dataset Generation Test"""

#modulation = "QPSK"
modulation = "BPSK"

samples = 10000
signal_length = 1024

save_dir = f"../data/{modulation.lower()}"
os.makedirs(save_dir, exist_ok=True)

split_config = {
    "train": 10000,
    "test": 5000,
    "validation": 2000 
}

for split, samples in split_config.items():
    save_dir = os.path.join("..","data",modulation.lower(),split)

    os.makedirs(save_dir, exist_ok=True)

    if modulation == "BPSK":
        clean_signals = np.empty((samples,signal_length), dtype=np.float32)
        
    elif modulation == "QPSK":
        clean_signals = np.empty((samples, signal_length // 2), dtype=np.complex64)

    for i in range(samples):
        
        bits = generate_bits(signal_length)
    
        if modulation == "BPSK":
            symbols = bpsk_mod(bits).astype(np.float32)
            
        elif modulation == "QPSK":
            symbols = qpsk_mod(bits).astype(np.complex64)

        clean_signals[i] = symbols
    
    save_path = os.path.join(save_dir, "clean.npy")
    np.save(save_path, clean_signals)